# VQCC Quantum Voodoo

This notebook mainly exists to help me understand what is going on in my own submission. When I first saw the circuits it looked like voodoo, and I delayed publishing it because Claude (the AI model I work with) was still explaining it to me. This is that explanation, written down.

It is also a working submission notebook. The hidden cells at the end pick the circuits, check every file against its SHA-256, run the unchanged official verifiers and write `submission.zip` with a receipt. Version 4 of this notebook scored **23.13505** on the public leaderboard (rank 78 of 1,667 when this text was written, 2026-10-30).

**How to read it.** Each section explains one technique in plain language, then shows a figure or animation drawn from our measurements. The code is hidden; click a cell to see it. Numbers carry the evidence tier they were measured at:

- *static model*: arithmetic on the official cost formula only;
- *local engine*: our fast re-implementation of the simulator, cross-checked against the official verifier;
- *local official verifier*: the unmodified competition verifier on a local build of kickmix;
- *observed public score*: the Kaggle leaderboard.

A number is never shown at a higher tier than it was measured at.

**Credit, up front.** Most of what is in these circuits is other people's work, and the credits section names them. The 889-qubit splice uses heliosli's edits on SomeJoint's decoder. The idea of "circuit surgery" is hengck23's. The 62-qubit inverse square root is yoyobar's, with Ahmed Berat Özer's improvement. Our own part is the splice itself, the 888-qubit release, the failure fix and the chemistry refit, each dated by this notebook's version history below.

## The version history is the record

Each version of this notebook submitted one portfolio from its Output tab. Kaggle keeps the versions, and the circuits dataset keeps its versions, so the dates below can be checked.

| Version | Date (UTC) | Inverse square root | DQI decoder | Chemistry | Public score |
| --- | --- | --- | --- | --- | --- |
| v1 | 2026-10-29, submitted 19:48 | 9a8627c81283 (busyaprime; mrmeows' design), 63 q | **7f3d64c1 (ours): the 889-qubit splice plus two flag cuts** | assembly of public parts | 22.96486 |
| v2 | 2026-10-30 00:47 | **35bfb6b5 (ours): −1 Toffoli on 9a8627**, 63 q | 7f3d64c1 | d7f8223d (Dmitrii Gluzdov) | 22.97237 |
| v3 | 2026-10-30 02:01 | 35bfb6b5 | **40fa4ebf (ours): 889 → 888 qubits** | d7f8223d | 22.94698 (one unlucky DQI shot; see section 4) |
| v4 | 2026-10-30 13:52 | 6dbb98b34c3b (Ahmed Berat Özer, on yoyobar's 62 q) | **2ed7ccf1 (ours): the failure fix** | **0d46767d (ours): refit of heliosli's weights** | **23.13505** |

Bold marks our own circuits. The 62-qubit inverse square root is not ours: yoyobar published the first one and Ahmed Berat Özer improved it (2026-10-30 11:12) before v4 used it.

In [ ]:
import base64, csv, glob, os, hashlib, json, shutil, sys, time, zipfile
from math import prod
from pathlib import Path

T0 = time.monotonic()
import numpy as np
import pandas as pd
from IPython.display import HTML, display

TASK_NAMES = {"invsqrt": "Inverse square root", "dqi": "DQI decoder", "chemistry": "Chemistry (dfthcsos)"}

def show(df, height=None):
    # A scrollable HTML table: readable on the notebook page, with live links
    html = df.to_html(index=False, render_links=True, escape=True, float_format=lambda x: f"{x:,.6g}")
    style = f"max-height:{height}px;overflow:auto;" if height else "overflow:auto;"
    display(HTML(f'<div style="{style}">{html}</div>'))

INPUT = Path(os.environ.get("VQCC_NB_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("VQCC_NB_WORK", "/kaggle/working"))
SUB = Path("/tmp/submission")   # assembled here; only submission.zip and check.json go to Output
WHEEL_GLOB = "kickmix-*-manylinux*x86_64*.whl"
# Organisers' reference costs (the score is the geometric mean of reference / ours over the three tasks)
REFERENCE = {"invsqrt": 2076774.233715282, "dqi": 24439463583672.93, "chemistry": 3355599588.3155375}
VERIFIER_TASK = {"invsqrt": "invsqrt", "dqi": "dqi", "chemistry": "dfthcsos"}

def find(pattern, root=INPUT):
    hits = sorted(Path(p) for p in glob.glob(f"{root}/**/{pattern}", recursive=True))  # follows symlinked mounts
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {root}")
    return hits

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()

def family_sha256(directory):
    # The dataset's chemistry family hash: SHA-256 of sorted '<relative path>\t<file sha256>\n' lines
    files = sorted(p for p in Path(directory).rglob("*") if p.is_file())
    text = "".join(f"{p.relative_to(directory).as_posix()}\t{sha256(p)}\n" for p in files)
    return hashlib.sha256(text.encode()).hexdigest()

In [ ]:
# Figures and animations live in the dataset's figures/ folder (made by scripts/build_quantum_voodoo_figures.py).
def figure(name, caption=""):
    hits = sorted(Path(p) for p in glob.glob(f"{INPUT}/**/figures/{name}", recursive=True))
    if not hits:
        print(f"[figure {name} is not in the attached dataset version]")
        return
    mime = "image/gif" if name.endswith(".gif") else "image/png"
    data = base64.b64encode(hits[0].read_bytes()).decode()
    cap = f'<figcaption style="color:#52514e;font-size:13px;margin-top:4px">{caption}</figcaption>' if caption else ""
    display(HTML(f'<figure style="margin:10px 0"><img src="data:{mime};base64,{data}" '
                 f'style="max-width:100%;height:auto" alt="{name}"/>{cap}</figure>'))

## 1. What the competition scores, and what a qubit is worth

There are three tasks. Each circuit gets a cost, and the score is the geometric mean, over the three tasks, of the organisers' reference cost divided by ours. A 1% cut in any one task's cost raises the score by about a third of a percent, whichever task it is.

The three cost formulas price qubits (Q) and Toffoli gates (T, the expensive non-Clifford gate) differently, so "save a qubit" means something different in each task:

| Task | Cost | Reference cost | v4 circuit | One qubit is worth |
| --- | --- | ---: | --- | --- |
| DQI decoder | R × Q² × T | 2.444 × 10¹³ | 888 q, 429,377 T per shot | about 967 Toffolis per shot (2T/Q), 0.23% of the cost |
| Chemistry | λ_eff × Q × T | 3.356 × 10⁹ | 430 q, 16,266 T | about 38 Toffolis (T/Q), 0.23% |
| Inverse square root | Q × (Toffolis + rotations/4) | 2.077 × 10⁶ | 62 q, 1,628 T-equivalents | about 26 T-equivalents (T/Q), 1.6% |

All figures in the last column are *static model*: arithmetic on the cost formula at the v4 circuit's size.

DQI is the odd one. Its cost has Q squared, so one qubit out of 888 is worth almost a thousand Toffolis per shot. Any change that frees a DQI qubit for fewer than about 960 extra Toffolis is a win. Most of this notebook is about DQI for that reason.

R is a penalty for failed shots. With no failures in 102,400 shots, R = 1.19126; each failed shot raises the DQI cost by about 0.46%. Section 4 is about R.

In [ ]:
figure("01_exchange_rates.png", "Static model from the official cost formulas at the v4 circuit sizes.")

## 2. Circuit surgery: 897 → 889 qubits by splicing two public circuits

The circuits are files of operations (a "netlist"), and anyone can read them. hengck23 suggested, in [a public discussion](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/discussion/753607), that an agent could cut a better module out of one circuit and paste it into another, keeping the splice only if a test says it helps. He called it **circuit surgery**. The idea and the name are his.

Two public DQI decoders came from the same ancestor (SomeJoint's b3354, 897 qubits):

- **SomeJoint's 53d8b2** ([v19](https://www.kaggle.com/code/somejoint/quantum-circuits-22-75-local-gm-897-qubit-dqi)) rebuilt two middle stages and saved about 9,800 Toffolis. Still 897 qubits.
- **heliosli's 4ee7a1** ([v11](https://www.kaggle.com/code/heliosli/the-dataset-is-now-public-now-enjoy)) went the other way: 889 qubits, but more Toffolis.

heliosli's trick concerns nine one-bit "pole" flags. Each records whether a denominator in the decoder was zero, and in SomeJoint's decoder all nine are held from the forward pass until the reverse pass needs them. They sit idle across the widest part of the circuit: 888 busy wires plus 9 parked flags makes 897. heliosli erases each flag right after it is made and recomputes it when it is needed again. Section 3 explains why erasing is cheap.

The two sets of edits touch different parts of the ancestor. We translated heliosli's edits, wire by wire, into SomeJoint's newer decoder: nine erasures, nine recompute blocks of 279 operations, and nine phase repairs. 2,529 operations in all. Every inserted block gets fresh wires and classical bits, so it cannot reuse a host wire by accident. A certified lifetime allocator then re-packs the qubits and proves that the re-packed file computes the same thing.

Result (local engine, official verifier cross-check): **889 qubits for +315 Toffolis per shot**, DQI cost −1.70%. Break-even was about 960 Toffolis per qubit; we paid about 39.

Two further cuts, following ideas hengck23 published, deleted zero tests that almost never fire, priced by measured chance rates (−560 and −165 Toffolis). The result, **7f3d64c1**, was −1.87% against SomeJoint's 53d8b2 and −1.23% against the best public DQI circuit of that day at the default draw (−0.78% like for like, since that circuit had one failed shot there). It went into v1.

The animation walks through the stages. Orange is SomeJoint's 53d8b2, blue the spliced circuit. Notice that the middle stages (convert, scan, unconvert) get narrower even though not one operation in them changed: they are narrower only because the poles are no longer parked across them.

In [ ]:
figure("02_splice_stages.gif", "Live qubits along each circuit, from the certified lifetime allocator (exact, structural). "
       "7f3d64c1 is aligned to 53d8b2 stage by stage.")

**How we knew a splice was safe.** Our formal version of the method treats a cut point's *interface* as everything that crosses it: which wires are live, what each one holds on every sampled shot, the sign, and any classical bits written before and read after. A donor module can replace a host module only if the interfaces match on both sides. Run over the public DQI circuits, the method rediscovered SomeJoint's own 53d8b2 from its ancestor, matching the cost to the last printed digit, and found nothing cheaper than 7f3d64c1. One lesson came from the measurement bits: every forward stage leaves hundreds of random measurement bits that only its mirror stage reads. So modules have to move in mirror pairs (extract with unextract), never alone.

## 3. Measurement-based uncomputation: why erasing is free, and what it costs later

The circuits are mostly reversible classical logic: each shot follows one row of bits plus a running sign (+ or −). Scratch wires must end at 0. There are two ways to clean a scratch wire that holds a = x1 AND x2:

1. **Uncompute it.** Run the Toffoli again. a XOR (x1 AND x2) = 0. Cost: one more Toffoli.
2. **Erase it by measurement (HMR).** Measure the wire sideways (in the X basis). The wire resets to 0 and the simulator records a random bit b. The catch: on rows where a was 1 and b came out 1, the sign of that row flips. That is a *phase debt*. It is repaid by a CZ on x1 and x2, applied only if b = 1. CZ is a Clifford gate, so it is free in every cost formula.

So erasing costs nothing now. The animation shows the four input rows as a ledger, the same format as our first lesson on signs.

In [ ]:
figure("03_hmr_ledger.gif", "Four input rows of a compute / use / erase / repay pattern. b is one example draw of the "
       "random bits; any draw ends in the same clean state.")

**What it costs later.** A measured-away value is gone. If the circuit needs it again, it must recompute it, which costs Toffolis. The record b cannot help: it is a coin flip, independent of the value (section 8 shows this is a property of the instruction set, not a design choice). So erasing pays when the qubit's price is higher than the recompute's. For DQI, that is almost always.

**889 → 888 (v3, ours).** After the splice, 7f3d64c1 touched 889 qubits in only one place: 82 short stretches inside the reverse pass (unextract), where the recomputed pole sat idle while other work ran. A census of every wire idle across those stretches found three blockers per epoch:

- the recomputed pole;
- a 12-wire "norm" value;
- a 3-wire scratch inside a small inverse.

Each could be released and rebuilt more cheaply. The pole is erased after its first use and recomputed just before its second, as [D = 0] with a small AND ladder. The norm, which we found is the exact quadratic Sq(a) + a·b + x¹¹·Sq(b), is erased and recomputed. The small inverse is done in place.

Total: +616 Toffolis per shot for one qubit worth about 970. A second edit noticed that one burst of 24 controlled swaps acts on registers whose contents are known whenever its control is on, so CNOTs can do its job (−192 T). The result, **40fa4ebf**, has 888 qubits and +424 Toffolis net, DQI cost −0.127% (local official verifier). The projection, 888²/889² × (1 + ΔT/T), predicted −0.1266%, and that is what was measured.

In [ ]:
figure("03_zoom_889_888.gif", "Zooming from the whole circuit into the unextract stage. Red dots: places where 7f3d64c1 is "
       "889 wide. Certified allocator profiles (exact, structural).")

Could 888 go to 887? No, not this way. In 40fa4ebf, 19 stretches have *no* idle wire at all: they are 24-bit multiplications that hold both inputs and a 24-wire product. A narrower multiply was priced and rejected (section 8).

## 4. Why a DQI score depends on a random draw, and why we fixed failures instead of re-rolling

The DQI verifier runs 102,400 shots. Each shot's input (how many errors, where) and every measurement's random bit come from a random generator **seeded by the SHA-256 of the circuit file**, plus a salt. The same file always gets the same draw, so a local run with the empty salt reproduces the public board exactly. Change one byte and you get a completely new draw. The final ranking uses a private salt nobody has.

The decoder lineage we build on fails on roughly 3 shots in a million, by chance. Whether a given draw contains such a shot is a lottery. A draw with one failure costs about 0.46% of DQI, about 0.15% of the overall score.

**What happened in v3.** 40fa4ebf passed with 0 failures at all four draws we tried locally. The public board, on the file's own draw, gave 22.94698 against our predicted 22.98206. The ratio cubed is 0.99543: exactly one failed shot. v2 had matched the board to 1e-5.

**Re-rolling.** Any byte change gets a new draw, so one could resubmit near-identical files until one draws zero failures. We decided not to. A lucky public draw says nothing about the private salt, which decides the prizes, and it makes the board less informative about the circuit. So we measured the failure rate instead, over 48 fixed salts (local engine), and went after its cause.

The animation shows those 48 draws for v3's circuit and for v4's fix. Same seeds, so each square compares the two circuits on the same draw.

In [ ]:
figure("04_draw_lottery.gif", "48 fixed salts, 102,400 shots each, local engine (sampled draws). Before: 13/48 draws have a "
       "failing shot. After: 6/48, one of them new (yellow ring).")

## 5. The failure fix (v4, ours): a chance zero meets an inverse

We collected all 13 failing shots of 40fa4ebf exactly and dumped the decoder's registers at every point where it inverts a 24-bit value. 11 of the 13 (85%) had the same cause:

- The decoder works in a finite field, where every non-zero value has an inverse. In the ordinary-node groups of the extraction (groups 0–7), it inverts 24-bit values 56 times per shot.
- On rare shots one of those values is **0 by chance**. The inverse circuit maps 0 to 0, so the information that "this was a pole" is lost. The decoder then finds a few spurious roots, gives up on the true error word and falls back to a default word, and the shot's phase comes out wrong.
- In 1,024 normal shots, no zero appeared at any of the 56 points. In the failing shots one always did.

As a sanity check (static model): if each of the 56 inputs were a uniform random 24-bit word, a 102,400-shot run would meet about 56 × 102,400 / 2²⁴ ≈ 0.34 zeros. We saw 11 such failures in 48 draws, 0.23 per draw.

In [ ]:
figure("05_chance_zero.gif", "The 13 failing shots (rows) at the 56 inverse calls of extract (columns). Red: a 24-bit "
       "zero is present. Exact register dumps of the local engine.")

**The fix.** SomeJoint's decoder already builds its later groups a different way, as *homogeneous chains* that carry a numerator and a denominator (N : D) instead of dividing. A chain cannot lose a pole halfway, because it never inverts. We measured both constructions group by group:

- ordinary node: about 399 + 1,316·g Toffolis for group g;
- chain: about 977 + 1,159·g.

With the pole flag dropped, the chain is cheaper from group 4 up. So groups 4–7 were rebuilt as chains with their poles dropped, and ported into 40fa4ebf.

Result, **2ed7ccf1**:

- **−1,460 Toffolis per shot** at 888 qubits;
- failing draws **13/48 → 6/48**;
- mean cost over the 48 draws −0.406% (local engine; official verifier PASS at the default salt).

One shot fails that did not fail before. That is the predicted price of dropping four poles, about 0.024 failures per verify. Keeping them would have cost far more than it saves. On the public board v4 scored 23.13505 against our predicted 23.13506, so the hosted draw had no DQI failure. That was luck of the draw, but now luck with better odds.

In [ ]:
figure("05_chain_vs_node.png", "Static model from measured per-group costs on 40fa4ebf. Blue: rebuilt in v4.")

## 6. Chemistry: refitting heliosli's weights (v4, ours, a small gain)

The chemistry task loads a factorized Hamiltonian into a circuit. Its cost is λ_eff × Q × T. λ_eff (an effective normalisation) sets how many times the circuit has to be repeated, so lowering it lowers the cost directly, with no circuit change. The weights must keep two accuracy residuals at or below 20 mHa, and a spectral gap above a fixed floor.

heliosli's published weights ([v21](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy)) were already right at all three limits: both residuals at 19.9994 mHa, the gap 1e-6 above its floor. A gradient step leaves the feasible set straight away. We minimised λ_eff while holding all three constraints *exactly*, with a projected L-BFGS that retracts onto the constraint surface after every step. This extends to three constraints anazemcev's constraint-manifold refit (Apache-2.0). Weights stuck exactly at zero were held there and released only when their gradient said so.

Result:

- λ_eff 48.3601 → **48.3007**;
- cost 338,248,977 → 337,833,376, **−0.12% chemistry**, about +0.04% on the overall score;
- word table and loaders unchanged;
- local official verifier PASS.

Honesty note: the experiment card's preregistered bar was a 0.2 drop in λ_eff, and it reached 0.06, so the card's verdict is **KILL**. The point used in v4 came from a longer diagnostic run beyond the registered 1,000-evaluation budget. We used it only as a free add-on. Getting further needs new words or a new factorization, not a better optimiser.

In [ ]:
figure("06_chem_refit.png", "λ_eff during the refit (official math re-implemented in float64; the end point was checked "
       "with the local official verifier).")

## 7. How the public inverse square root got from 63 to 62 qubits (yoyobar, then Ahmed Berat Özer)

Not our work, and a lovely piece of it. On 2026-10-29 our own [published note](https://doi.org/10.5281/zenodo.23781241) argued that the best 63-qubit circuit (busyaprime's 9a8627, on mrmeows' design) was optimal for its architecture on the axes we tested. Width was not one of those axes. **yoyobar** has since published a 62-qubit circuit, a568bc ([LB 22.99 | invsqrt 101 - 62 qubits](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits); we pulled version 4, run 2026-10-30 04:41 UTC). yoyobar's notebook says it was developed with GPT-6 Astra, reworking the encoded lookup, the normalization and the tail schedule, and building on busyaprime's circuit, mrmeows / team selenium's design and our Friends notebook. heliosli later republished it ([notebook](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy)) with credit to yoyobar. We had believed 63 → 62 was blocked: a flag q23 = [x < 2²⁰] holds information that no other live wire has. The 62-qubit circuit kept that flag and took one wire out of **every other stage** instead:

- **Coefficient lookup.** The products go straight into the wires that become output words, so 5 temporaries are alive at once instead of 32. One wire fewer, for 5 more Toffolis.
- **Later stages.** Fresh helper wires are replaced by in-place XORs onto wires that are already live, and by classical logic on measurement bits: if this recorded bit is 1, apply that fix-up.
- **Net.** −1 qubit for about +4.5 Toffoli-equivalents, against a break-even of about 26. Cost −1.32%, with the same function, the same architecture and the same worst-case error. We checked this exactly over all 3,133,452 inputs.

**Ahmed Berat Özer** then removed one more Toffoli from yoyobar's file ([v15](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression), 2026-10-30 11:12). One controlled swap needs the AND q19·(q6 ⊕ q41). That AND turns out to equal an XOR of wires already live at that point, on every legal input. So it can be computed with CNOTs into a free wire and erased by measurement, with no Toffoli. Our v4 uses Özer's file, credited.

Our follow-up searched all 1,501 Toffoli-class gates of Özer's circuit for another such identity, exactly, over all inputs. There is none (section 8).

*A correction, with apologies:* an earlier version of this notebook credited heliosli; the circuit is yoyobar's. heliosli republished it with credit to yoyobar, and we mistook that republication for the original. Thank you to both, and sorry for the mix-up.

In [ ]:
figure("07_invsqrt_stages.gif", "Live qubits per operation for 9a8627 (63 q) and yoyobar's a568bc (62 q), aligned stage by "
       "stage at five anchors where the two circuits hold identical values. Exact over all inputs; widths structural.")

## 8. What did not work, and why

Failures are kept with the same care as passes. Each line below was preregistered with a gate and then run.

| Idea | Verdict | Why (tier) |
| --- | --- | --- |
| Replace the reverse pass's recomputation with the forward pass's measurement record | **Kill: 0 Toffolis replaceable** | In kickmix the only op that writes a bit from a qubit is HMR, and it writes `b ^= random`, with the qubit's value going only into the sign. **The record is random bits, independent of the data** (exact, from the simulator source), confirmed by a sampled test: identical records in 12 batches, different recomputed values in 256 of 256 lanes. |
| Cheaper convert/unconvert chains | **Kill** | Every priced alternative is ≥ 160k T-equivalents; today's chain is the cheapest. Replaying measured values is ruled out by the fact above, and an in-place multiply needs a field inverse (static model). |
| 888 → 887 with a narrower 24-bit multiply | **Kill** | The last product bit needs all 24 input bits, the other 23 product bits and the multiplier live at once (exact lemma). Narrower designs cost ≥ 118 Toffolis per call against a 26.8 budget. |
| Graft hengck23's cheaper synthesis stage into ours | **Incompatible** | The interfaces differ by a nonlinear map (projective vs affine field elements); the adaptor would be the very inversions his design avoids (sampled state comparison). |
| Delete all 17 low folds (the full form of hengck23's cut) | **Kept 9** | Odd folds fire by chance about 2.4e-7 per shot, which is near break-even: a build with all 17 removed (on heliosli's 4ee7) lost 0.58% on its own draw (local engine). |
| Splice search over all 21 public DQI circuits | **Nothing cheaper** | 7f3d64c1's modules are the cheapest in their compatibility class (sampled + official verifier). |
| More Özer-style edits in the 62-qubit inverse square root | **Kill** | 1,501 gates checked over all inputs: no affine form. 65 apparent hits were artefacts of one random branch, refuted on others (exact per seed). |
| Chemistry weight refit to the 0.2 bar | **Kill (kept as add-on)** | Converged to a near-KKT point 0.06 below the start (local official verifier). |

The first row is the instructive one, and it is the reason section 3 says "the record cannot help". Measurement in these circuits does not *read* a value. It destroys the value and leaves a coin flip behind. Anything you erase by measurement, you pay for again if you need it.

## 9. Credits

The circuits in this notebook stand on public work released under Apache-2.0. Thank you to everyone who publishes their circuits.

- **SomeJoint (Agniv Banerjee's team)**: the DQI decoder all our DQI circuits are built on (b3354f60e459, [53d8b2edf251 v19](https://www.kaggle.com/code/somejoint/quantum-circuits-22-75-local-gm-897-qubit-dqi)), including the homogeneous-chain construction we extended in v4.
- **heliosli**: the pole release that made 889 possible ([4ee7a17b4cbc v11](https://www.kaggle.com/code/heliosli/the-dataset-is-now-public-now-enjoy)); the republication of yoyobar's 62-qubit circuit, with credit ([notebook](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy)); the chemistry weights we refit (7c6c875a7516, v21) and chemistry loaders.
- **hengck23**: the idea and name of circuit surgery and the two flag-cut ideas ([discussion](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/discussion/753607)); his 889-qubit decoder 7a378f ([notebook](https://www.kaggle.com/code/hengck23/dqi-kmb-at-59-56854)), integrated and audited by **Kirill Tymchishin** ([v25](https://www.kaggle.com/code/kirilltym/quantum-circuits-exact-gauge-5-word-qrom)).
- **yoyobar**: the first public 62-qubit inverse square root ([a568bc793481](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits), developed with GPT-6 Astra), on which Özer's file is built.
- **Ahmed Berat Özer**: the 62-qubit inverse square root in v4 ([6dbb98b34c3b v15](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression)), the affine-identity edit explained in section 7, and the chemistry weights in v1.
- **Akmal Xodarev (busyaprime)** and **mrmeows**: the 63-qubit inverse square root in v1–v3 ([9a8627c81283 v10](https://www.kaggle.com/code/busyaprime/lb-22-72431-22x-cheaper-than-google-s-circuits)).
- **Dmitrii Gluzdov** ([d7f8223d0f9b v37](https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source)) and **Atakan Aldemir**: chemistry loaders across the versions (v2 and v3 use Gluzdov's whole family), and Atakan Aldemir's word table.
- **anazemcev**: the constraint-manifold refit our chemistry refit extends.
- **Google Quantum AI**: the competition, kickmix and the official verifiers.

Every public circuit, with its source notebook, version and SHA-256, is in our dataset [VQCC public circuits (Apache-2.0)](https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits). The circuits marked "ours" are by manwithacat (James Barlow), made with substantial help from Claude (Anthropic), which wrote most of the tooling and analysis under my direction. My contribution was choosing the questions, approving each experiment and checking the claims.

## The submission this notebook makes

The hidden cells below are the same verified path as every earlier version:

1. pick the cheapest public circuit per task from the public dataset's manifest;
2. replace the DQI and chemistry picks with ours from `private.json` in this notebook's dataset, checking every file's SHA-256;
3. install our kickmix fork offline;
4. run the unmodified official verifiers at full settings;
5. write `submission.zip` only if all three pass.

The receipt's gates check that the files are the listed ones, that all three passed, and that every cost matches the dataset's record of the same verifier on the same bytes.

In [ ]:
manifest = find("manifest.csv")[0]
DATA = manifest.parent
rows = list(csv.DictReader(open(manifest, newline="")))
print(f"manifest: {manifest} ({len(rows)} circuits, sha256 {sha256(manifest)[:16]}...)")

picks = {}
for task in REFERENCE:
    scored = [r for r in rows if r["task"] == task and r["local_task_score"]]
    scored.sort(key=lambda r: float(r["local_task_score"]))
    picks[task] = scored[0]
    print(f"{task:9} {len(scored):2} candidates; cheapest {scored[0]['sha12']} "
          f"cost {float(scored[0]['local_task_score']):.6e}")

# Chemistry: a verified assembly of public parts replaces the best single family only if it is cheaper
ASSEMBLY = None
best_json = DATA / "parts" / "assembly_best.json"
if best_json.is_file():
    a = json.loads(best_json.read_text())
    single = float(picks["chemistry"]["local_task_score"])
    print(f"parts/assembly_best.json: verified {a['verified_cost']:.6e} ({a['verify_date']}) "
          f"vs best single family {picks['chemistry']['sha12']} {single:.6e}")
    if a.get("verified_status") == "PASS" and float(a["verified_cost"]) < single:
        ASSEMBLY = a

shutil.rmtree(SUB, ignore_errors=True)
SUB.mkdir(parents=True)
for task, r in picks.items():
    if task == "chemistry" and ASSEMBLY:
        fam = SUB / "dfthcsos"
        fam.mkdir()
        files = [(ASSEMBLY["npz"]["dataset_path"], ASSEMBLY["npz"]["sha256"], "submission.npz")]
        files += [(s["dataset_path"], s["file_sha256"], f"angle_k_{s['slot']}.kmb") for s in ASSEMBLY["slots"]]
        for path, digest, name in files:
            assert sha256(DATA / path) == digest, f"{path} does not match parts/assembly_best.json"
            shutil.copyfile(DATA / path, fam / name)
        authors = sorted({s["author"] for s in ASSEMBLY["slots"]})
        picks["chemistry"] = {
            "task": "chemistry", "sha12": "assembled", "sha256": family_sha256(fam),
            "qubits": ASSEMBLY["qubits"], "local_task_score": str(ASSEMBLY["verified_cost"]),
            "author": "assembled: loaders " + ", ".join(authors) + "; weights " + ASSEMBLY["npz"]["author"],
            "notebook_title": "assembly of public chemistry parts (parts/assembly_best.json)",
            "source_notebook_url": "https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits",
            "source_notebook_version": "", "release_date": ASSEMBLY["verify_date"],
            "core_author": ASSEMBLY["word_table_origin"]["author"],
            "change_summary": f"cheapest public loader per slot with the best weights on word table "
                              f"{ASSEMBLY['table_key']}", "also_found_in": ""}
        print(f"chemistry: assembled from {len(files)} dataset files, every SHA-256 checked")
    elif task == "chemistry":
        src = DATA / "circuits" / "chemistry" / r["sha12"]
        assert family_sha256(src) == r["sha256"], f"chemistry family {r['sha12']} does not match the manifest"
        shutil.copytree(src, SUB / "dfthcsos")
    else:
        src = DATA / "circuits" / task / f"{r['sha12']}.kmb"
        assert sha256(src) == r["sha256"], f"{task} {r['sha12']} does not match the manifest"
        shutil.copyfile(src, SUB / f"{task}.kmb")
print("assembled and hash-checked:", sorted(p.name for p in SUB.iterdir()))

pd.set_option("display.max_colwidth", 80)
display(pd.DataFrame([{"task": t, "candidates": sum(1 for x in rows if x["task"] == t and x["local_task_score"]),
                       "picked": r["sha12"], "qubits": r.get("qubits"),
                       "local cost": float(r["local_task_score"]), "author": r["author"],
                       "sha256 checked": True} for t, r in picks.items()]).set_index("task"))

## Replace picks with our private circuits

In [ ]:
private_json = find("private.json")[0]
PRIVATE_PICKS = json.loads(private_json.read_text())
for task, PRIV in PRIVATE_PICKS.items():
    src = private_json.parent / PRIV["path"]
    public = picks[task]
    if task == "chemistry":  # a family directory: hashed like the dataset's families
        assert family_sha256(src) == PRIV["sha256"], "private chemistry family does not match private.json"
        shutil.rmtree(SUB / "dfthcsos", ignore_errors=True)
        shutil.copytree(src, SUB / "dfthcsos")
    else:
        assert sha256(src) == PRIV["sha256"], f"private {task} does not match private.json"
        shutil.copyfile(src, SUB / f"{task}.kmb")
    picks[task] = {
        "task": task, "sha12": PRIV["sha12"], "sha256": PRIV["sha256"], "qubits": PRIV["qubits"],
        "local_task_score": PRIV["local_task_score"], "author": "manwithacat (private)",
        "notebook_title": "private candidate: " + PRIV["change"], "source_notebook_url": "",
        "source_notebook_version": "", "release_date": "", "core_author": PRIV["base"],
        "change_summary": PRIV["change"], "also_found_in": ""}
    print(f"{task}: private {PRIV['sha12']} ({PRIV['qubits']} qubits, local cost "
          f"{float(PRIV['local_task_score']):.6e}) replaces public {public['sha12']} "
          f"({float(public['local_task_score']):.6e}): "
          f"{float(PRIV['local_task_score']) / float(public['local_task_score']) - 1:+.3%}")

In [ ]:
tool_dirs = [Path(p).parent for root in ("/kaggle/usr/lib", "/kaggle/input")
             for p in sorted(glob.glob(f"{root}/**/kickmix_tools.py", recursive=True))]
assert tool_dirs, "attach the kickmix_tools utility script"
sys.path.insert(0, str(tool_dirs[0]))
import kickmix_tools as kt
print("kickmix_tools", kt.__version__, "from", tool_dirs[0])

wheel_dir = find(WHEEL_GLOB)[0].parent
FORK_DIR = kt.install(str(wheel_dir), target="/tmp/kickmix_fork")
FORK_PROVENANCE = kt.provenance()
print("fork provenance:", FORK_PROVENANCE)
print("competition data:", kt.competition_dir())

In [ ]:
PATHS = {"invsqrt": SUB / "invsqrt.kmb", "dqi": SUB / "dqi.kmb", "chemistry": SUB / "dfthcsos"}
results, check_rows = {}, []
for task, circuit in PATHS.items():
    r = kt.verify(VERIFIER_TASK[task], circuit, build="fork")
    results[task] = r
    res = r["result"]
    listed = float(picks[task]["local_task_score"])
    check_rows.append({
        "task": task, "circuit": picks[task]["sha12"], "status": res["status"],
        "cost": res["score"], "manifest cost": listed, "cost / manifest": res["score"] / listed,
        "times cheaper than reference": REFERENCE[task] / res["score"],
        "qubits": res.get("num_qubits"), "Toffolis (mean)": res.get("toffoli_count"),
        "value errors": res.get("value_errors") if task == "dqi" else None,
        "phase errors": res.get("phase_errors") if task == "dqi" else None,
        "wall s": round(r["wall_seconds"], 1),
        "peak MB": round(r["peak_pss_sum_mb"] or r["peak_rss_sum_mb"] or 0)})
checks = pd.DataFrame(check_rows).set_index("task")
FORMATS = {"cost": "{:.6e}", "manifest cost": "{:.6e}", "cost / manifest": "{:.9f}",
           "times cheaper than reference": "{:.4f}", "qubits": "{:.0f}", "Toffolis (mean)": "{:,.1f}",
           "value errors": "{:.0f}", "phase errors": "{:.0f}", "wall s": "{:.1f}", "peak MB": "{:,.0f}"}
shown = checks.copy()
for col, fmt in FORMATS.items():  # value/phase errors exist for DQI only; blank cells show as —
    shown[col] = [fmt.format(v) if pd.notna(v) else "—" for v in checks[col]]
display(shown)
print("— = not applicable: only the DQI verifier counts value and phase errors.")
failed = [t for t, r in results.items() if r["result"]["status"] != "PASS"]
assert not failed, f"not writing a submission: {failed} did not pass"

ratios = {t: REFERENCE[t] / results[t]["result"]["score"] for t in REFERENCE}
gm = prod(ratios.values()) ** (1 / 3)
print(f"our local geometric mean: {gm:.5f}  (the leaderboard re-measures with its own samples)")
checks.to_csv(WORK / "checks.csv")

In [ ]:
members = [("invsqrt.kmb", SUB / "invsqrt.kmb"), ("dqi.kmb", SUB / "dqi.kmb")]
members += [(f"dfthcsos/{p.relative_to(SUB / 'dfthcsos').as_posix()}", p)
            for p in sorted((SUB / "dfthcsos").rglob("*")) if p.is_file()]
with zipfile.ZipFile(WORK / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name, path in members:
        z.write(path, name)
(WORK / "check.json").write_text(json.dumps({
    "manifest_sha256": sha256(manifest), "submission_sha256": sha256(WORK / "submission.zip"),
    "geometric_mean_local": gm, "fork": FORK_PROVENANCE,
    "picks": {t: {k: picks[t][k] for k in ("sha256", "author", "notebook_title", "source_notebook_url",
                                            "release_date", "local_task_score")} for t in picks},
    "chemistry_assembly": ASSEMBLY,
    "results": {t: r["result"] for t, r in results.items()}}, indent=1, default=str))
shutil.rmtree(SUB)
print("wrote", WORK / "submission.zip", "and", WORK / "check.json")

## Receipt

One table that says what was submitted and why it can be trusted. Each gate is a check this run made, not a claim copied from elsewhere: the file is the one the manifest lists (by SHA-256; for an assembled chemistry entry, every file is the one `parts/assembly_best.json` lists), the unchanged official verifier passed it, and its cost matches the dataset's record of the same verifier on the same bytes. Saved as `receipt.json`.

In [ ]:
WALL = time.monotonic() - T0
receipt = pd.DataFrame([{
    "task": TASK_NAMES[t], "circuit": picks[t]["sha12"], "sha256": picks[t]["sha256"][:16] + "…",
    "published by": picks[t]["author"], "version": picks[t]["source_notebook_version"],
    "verdict": results[t]["result"]["status"],
    "times cheaper than reference": REFERENCE[t] / results[t]["result"]["score"],
    "link": picks[t]["source_notebook_url"]} for t in REFERENCE])
show(receipt)
CHEM_PARTS = None
if ASSEMBLY:
    print("Chemistry was assembled from public parts: each slot's loader, the weights and the word table's origin")
    CHEM_PARTS = pd.DataFrame(
        [{"part": f"slot {s['slot']} loader", "file": s["dataset_path"], "sha256": s["file_sha256"][:16] + "…",
          "published by": s["author"], "version": s.get("notebook_version", ""),
          "Toffolis per shot": s.get("toffoli_per_shot_source"), "link": s["notebook"]} for s in ASSEMBLY["slots"]]
        + [{"part": "weights (submission.npz)", "file": ASSEMBLY["npz"]["dataset_path"],
            "sha256": ASSEMBLY["npz"]["sha256"][:16] + "…", "published by": ASSEMBLY["npz"]["author"],
            "version": ASSEMBLY["npz"].get("notebook_version", ""), "Toffolis per shot": None,
            "link": ASSEMBLY["npz"]["notebook"]},
           {"part": f"word table {ASSEMBLY['table_key']} (origin)", "file": "parts/word_tables.csv",
            "sha256": "", "published by": ASSEMBLY["word_table_origin"]["author"], "version": "",
            "Toffolis per shot": None, "link": ASSEMBLY["word_table_origin"]["notebook"]}])
    show(CHEM_PARTS)
rel = {t: abs(results[t]["result"]["score"] / float(picks[t]["local_task_score"]) - 1) for t in REFERENCE}
GATES = {
    "G1 every file matches its listed SHA-256": True,  # asserted when the files were assembled
    "G2 the official verifier passed all three": all(r["result"]["status"] == "PASS" for r in results.values()),
    "G3 every cost within 1e-9 of the manifest record": max(rel.values()) <= 1e-9,
    "G4 notebook wall time under 2 hours": WALL <= 7200,
}
show(pd.DataFrame([{"gate": g, "result": "pass" if ok else "FAIL"} for g, ok in GATES.items()]))
print(f"geometric mean {gm:.5f}; wall time {WALL / 60:.1f} min. G3 means each cost was reproduced exactly: the "
      f"verifier seeds every shot from the file's SHA-256, so the same bytes always give the same cost.")
print(f"Predicted public leaderboard score: {gm:.5f}. The public board uses the same fixed draw per file, so it "
      f"should match to about 1e-5; the final ranking re-draws DQI shots with a private salt.")

# Headroom: how close each pick is to the verifier's pass/fail limits (from this run's verifier output)
E1_MAX = E2_MAX = 0.020          # Ha, both chemistry accuracy gates
GAP_FLOOR = 3.914898             # strict E_gap floor in the dfthcsos verifier
head = []
inv = results["invsqrt"]["result"]
w = inv.get("worst_error_over_tolerance")
if w is not None:
    head.append({"task": TASK_NAMES["invsqrt"], "check": "worst error / tolerance (must stay below 1)",
                 "value": f"{w:.5f}", "margin": f"{(1 - w) * 100:.3f}% of tolerance",
                 "tight": (1 - w) < 0.005})
dq = results["dqi"]["result"]
head.append({"task": TASK_NAMES["dqi"], "check": "failed shots on this file's draw (value / phase)",
             "value": f"{dq.get('value_errors')} / {dq.get('phase_errors')}",
             "margin": f"repetition factor R = {dq.get('repetition_factor', float('nan')):.5f}; "
                       "each extra failure costs about 0.46% of DQI cost",
             "tight": (dq.get("value_errors") or 0) + (dq.get("phase_errors") or 0) > 0})
ch = results["chemistry"]["result"].get("metrics", {})
for key, lim in (("error_1", E1_MAX), ("error_2", E2_MAX)):
    if key in ch:
        m = (lim - ch[key]) * 1000
        head.append({"task": TASK_NAMES["chemistry"], "check": f"{key} (must stay ≤ 20 mHa)",
                     "value": f"{ch[key] * 1000:.5f} mHa", "margin": f"{m:.5f} mHa", "tight": m < 0.01})
if "E_gap" in ch:
    m = ch["E_gap"] - GAP_FLOOR
    head.append({"task": TASK_NAMES["chemistry"], "check": "E_gap (must stay above 3.914898)",
                 "value": f"{ch['E_gap']:.7f}", "margin": f"{m:.2e}", "tight": m < 1e-4})
hd = pd.DataFrame(head)
hd["status"] = ["tight" if t else "comfortable" for t in hd.pop("tight")]
print("Headroom: how close each pick is to failing the verifier. 'tight' means a tiny change in the file, the "
      "verifier or floating point could flip it.")
show(hd)

# DQI final-salt risk for the picked file, from the DQI Salt Check notebook's output when it covers this file
SALT = None
for f in sorted(glob.glob(f"{INPUT}/**/dqi_salt_check_summary_*.csv", recursive=True)):
    for r in csv.DictReader(open(f, newline="")):
        if r.get("circuit") == picks["dqi"]["sha12"]:
            SALT = dict(r, file=Path(f).name)
if SALT:
    print(f"DQI final-salt risk for {picks['dqi']['sha12']} ({SALT['runs']} salts in {SALT['file']}): "
          f"errors per salt {SALT['errors per salt']}; P(0 failures) {SALT['P(0 err)']}; expected GM factor "
          f"{SALT['E[GM factor]']} (95% {SALT['GM factor 95%']}).")
else:
    print(f"DQI final-salt risk for {picks['dqi']['sha12']}: not measured yet (this file is not in the attached "
          f"DQI Salt Check output).")
if not GATES["G3 every cost within 1e-9 of the manifest record"]:
    print("G3 failed: this run's verifier disagrees with the manifest, so treat the local score with care.")
(WORK / "receipt.json").write_text(json.dumps({
    "gates": GATES, "geometric_mean_local": gm, "headroom": head, "dqi_salt_check": SALT, "wall_seconds": WALL, "relative_cost_difference": rel,
    "submission_sha256": sha256(WORK / "submission.zip"), "manifest_sha256": sha256(manifest),
    "rows": receipt.to_dict(orient="records"),
    "chemistry_parts": CHEM_PARTS.to_dict(orient="records") if CHEM_PARTS is not None else None},
    indent=1, default=str))